# Train 24h volatility models (Colab)

**Yes — this notebook uses all three:**

| Technique | How |
|---|---|
| Hyperparameters | Optuna (Bayesian search) on **LightGBM** and **XGBoost** |
| Validation | `TimeSeriesSplit` on the *train* window only (no shuffle, holdout untouched) |
| Ensemble | `VotingRegressor` (average) and `StackingRegressor` (Ridge meta-learner) |
| Baselines | Persistence (`vol_24h_hist`), Linear Regression, Random Forest |

Winner = lowest holdout **RMSE**. Also report MAE, R², directional accuracy.

Upload `data/features.parquet` first. After Run all: download `models/model.joblib`, `models/feature_columns.json`, and `mlruns.zip` into the local project. Then we can open MLflow UI locally.


In [1]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q pandas numpy scikit-learn lightgbm xgboost optuna mlflow joblib pyarrow

import json
import warnings
from pathlib import Path

import joblib
import lightgbm as lgb
import mlflow
import mlflow.sklearn
import numpy as np
import optuna
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor, StackingRegressor, VotingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
print("IN_COLAB =", IN_COLAB)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 94.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 125.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 133.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.2/216.2 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# Keep modest so Colab finishes. Raise later if you want a longer search.
N_TRIALS = 20
N_CV_SPLITS = 3
HOLDOUT_DAYS = 60
RANDOM_STATE = 42
EXPERIMENT = "crypto-vol-24h"
MODEL_NAME = "crypto_vol_24h"

FEATURE_COLUMNS = [
    "log_return", "abs_return", "vol_6h", "vol_24h_hist", "vol_72h",
    "vol_term_structure", "candle_range", "bb_width", "bb_pct", "rsi_14",
    "sma_ratio", "ema_ratio", "log_volume", "volume_z", "trades_z",
    "taker_buy_ratio", "shock_volume", "hour", "dow", "symbol_id",
]
TARGET = "vol_24h"
TS = "event_timestamp"


In [3]:
CANDIDATES = [
    Path("data/features.parquet"),
    Path("/content/features.parquet"),
    Path("/content/data/features.parquet"),
]

def load_features():
    for path in CANDIDATES:
        if path.exists():
            print("Loaded", path)
            return pd.read_parquet(path), str(path)
    if IN_COLAB:
        from google.colab import files
        print("Upload data/features.parquet")
        uploaded = files.upload()
        name = next(iter(uploaded))
        return pd.read_parquet(name), name
    raise FileNotFoundError("features.parquet not found. Run src.eda_feature_eng locally first.")

df, src = load_features()
df[TS] = pd.to_datetime(df[TS], utc=True)
df = df.dropna(subset=FEATURE_COLUMNS + [TARGET]).sort_values(TS).reset_index(drop=True)
print(src, "rows", len(df), "range", df[TS].min(), "→", df[TS].max())


Upload data/features.parquet


Saving features.parquet to features.parquet
features.parquet rows 51456 range 2025-02-27 15:00:00+00:00 → 2026-08-17 14:00:00+00:00


In [4]:
cutoff = df[TS].max() - pd.Timedelta(days=HOLDOUT_DAYS)
train_df = df.loc[df[TS] < cutoff].copy()
test_df = df.loc[df[TS] >= cutoff].copy()
X_train, y_train = train_df[FEATURE_COLUMNS], train_df[TARGET]
X_test, y_test = test_df[FEATURE_COLUMNS], test_df[TARGET]
print("train", len(train_df), "holdout", len(test_df), "cutoff", cutoff)


def metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    true_dir = np.sign(np.diff(y_true))
    pred_dir = np.sign(np.diff(y_pred))
    mask = true_dir != 0
    dir_acc = float((true_dir[mask] == pred_dir[mask]).mean()) if mask.sum() else float("nan")
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
        "directional_accuracy": dir_acc,
    }


def cv_rmse(model_fn, n_splits=N_CV_SPLITS):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    scores = []
    Xv, yv = X_train.reset_index(drop=True), y_train.reset_index(drop=True)
    for tr, va in tscv.split(Xv):
        model = model_fn()
        model.fit(Xv.iloc[tr], yv.iloc[tr])
        pred = model.predict(Xv.iloc[va])
        scores.append(np.sqrt(mean_squared_error(yv.iloc[va], pred)))
    return float(np.mean(scores))


train 45692 holdout 5764 cutoff 2026-06-18 14:00:00+00:00


## 1. Baselines (no Optuna)


In [5]:
leaderboard = {}
fitted = {}

persist = X_test["vol_24h_hist"].to_numpy()
leaderboard["persistence"] = metrics(y_test, persist)
print("persistence", leaderboard["persistence"])

lin = LinearRegression()
lin.fit(X_train, y_train)
fitted["linear"] = lin
leaderboard["linear"] = metrics(y_test, lin.predict(X_test))
print("linear", leaderboard["linear"])

rf = RandomForestRegressor(
    n_estimators=200, max_depth=12, min_samples_leaf=5,
    random_state=RANDOM_STATE, n_jobs=-1,
)
rf.fit(X_train, y_train)
fitted["random_forest"] = rf
leaderboard["random_forest"] = metrics(y_test, rf.predict(X_test))
print("random_forest", leaderboard["random_forest"])


persistence {'rmse': 0.0020428514493930007, 'mae': 0.0013921996002122113, 'r2': 0.13119589707425527, 'directional_accuracy': 0.7537740760020822}
linear {'rmse': 0.0018808312565507996, 'mae': 0.0014075268029607908, 'r2': 0.26354204771227585, 'directional_accuracy': 0.7955925733125109}
random_forest {'rmse': 0.0017861631448379653, 'mae': 0.0012656143597101904, 'r2': 0.33581274593787647, 'directional_accuracy': 0.7836196425472844}


## 2. Optuna hyperparameter search (time-series CV)

Search happens **only on the train window**. The last 60 days stay locked as the test set.


In [6]:
def tune_lgbm(n_trials=N_TRIALS):
    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 200, 500),
            "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.12, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 16, 64),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 80),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
            "random_state": RANDOM_STATE,
            "n_jobs": -1,
            "verbosity": -1,
        }
        return cv_rmse(lambda: lgb.LGBMRegressor(**params))

    study = optuna.create_study(direction="minimize", study_name="lgbm")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    best = lgb.LGBMRegressor(**{**study.best_params, "random_state": RANDOM_STATE, "n_jobs": -1, "verbosity": -1})
    best.fit(X_train, y_train)
    return best, study

lgbm, lgbm_study = tune_lgbm()
fitted["lightgbm"] = lgbm
leaderboard["lightgbm"] = metrics(y_test, lgbm.predict(X_test))
print("lgbm best params", lgbm_study.best_params)
print("lightgbm", leaderboard["lightgbm"])


  0%|          | 0/20 [00:00<?, ?it/s]

lgbm best params {'n_estimators': 457, 'learning_rate': 0.03710605072792527, 'num_leaves': 16, 'min_child_samples': 29, 'subsample': 0.8473316434494441, 'colsample_bytree': 0.6912519827930352, 'reg_lambda': 0.0014718613502238006}
lightgbm {'rmse': 0.0017303320604809987, 'mae': 0.0012489229586734197, 'r2': 0.37668554512333674, 'directional_accuracy': 0.8042686100989068}


In [7]:
def tune_xgb(n_trials=N_TRIALS):
    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 200, 500),
            "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.12, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 8),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 12),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
            "random_state": RANDOM_STATE,
            "n_jobs": -1,
            "tree_method": "hist",
        }
        return cv_rmse(lambda: xgb.XGBRegressor(**params))

    study = optuna.create_study(direction="minimize", study_name="xgb")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    best = xgb.XGBRegressor(**{**study.best_params, "random_state": RANDOM_STATE, "n_jobs": -1, "tree_method": "hist"})
    best.fit(X_train, y_train)
    return best, study

xgbr, xgb_study = tune_xgb()
fitted["xgboost"] = xgbr
leaderboard["xgboost"] = metrics(y_test, xgbr.predict(X_test))
print("xgb best params", xgb_study.best_params)
print("xgboost", leaderboard["xgboost"])


  0%|          | 0/20 [00:00<?, ?it/s]

xgb best params {'n_estimators': 243, 'learning_rate': 0.020277694560104875, 'max_depth': 5, 'min_child_weight': 7, 'subsample': 0.8068200099991135, 'colsample_bytree': 0.663636564169796, 'reg_lambda': 1.1009700520286325}
xgboost {'rmse': 0.0017280378831735286, 'mae': 0.001264259150962467, 'r2': 0.3783373042566336, 'directional_accuracy': 0.8032274856845393}


## 3. Ensemble learning


In [8]:
from sklearn.model_selection import KFold

# 1. Voting Regressor
vote = VotingRegressor(
    estimators=[("lgbm", lgbm), ("xgb", xgbr), ("rf", rf)],
    weights=[2, 2, 1],
    n_jobs=-1,
)
vote.fit(X_train, y_train)
fitted["voting"] = vote
leaderboard["voting"] = metrics(y_test, vote.predict(X_test))
print("voting", leaderboard["voting"])

# 2. Stacking Regressor (Fixed with KFold)
stack = StackingRegressor(
    estimators=[("lgbm", lgbm), ("xgb", xgbr), ("rf", rf)],
    final_estimator=Ridge(alpha=1.0),
    cv=KFold(n_splits=N_CV_SPLITS, shuffle=False),
    n_jobs=-1,
)
stack.fit(X_train, y_train)
fitted["stacking"] = stack
leaderboard["stacking"] = metrics(y_test, stack.predict(X_test))
print("stacking", leaderboard["stacking"])

# Leaderboard display
board = pd.DataFrame(leaderboard).T.sort_values("rmse")
display(board)
winner_name = board.index[0]
print("WINNER =", winner_name)
winner = fitted.get(winner_name)

voting {'rmse': 0.0017260704844713815, 'mae': 0.0012492812107575825, 'r2': 0.37975204424230236, 'directional_accuracy': 0.8065243796633698}
stacking {'rmse': 0.002301275833345167, 'mae': 0.001934856605609928, 'r2': -0.1025178833082474, 'directional_accuracy': 0.8079125455491931}


,rmse,mae,r2,directional_accuracy
voting,0.001726,0.001249,0.379752,0.806524
xgboost,0.001728,0.001264,0.378337,0.803227
lightgbm,0.001730,0.001249,0.376686,0.804269
random_forest,0.001786,0.001266,0.335813,0.783620
linear,0.001881,0.001408,0.263542,0.795593
persistence,0.002043,0.001392,0.131196,0.753774
stacking,0.002301,0.001935,-0.102518,0.807913


WINNER = voting


## 4. Per-symbol holdout + MLflow registry


In [9]:
if winner is None:
    raise RuntimeError("Winner is a baseline without a dumpable model (persistence). Pick the next row.")

y_hat = np.asarray(winner.predict(X_test), dtype=float)
by_symbol = {}
tmp = test_df.copy()
tmp["_yt"] = y_test.to_numpy()
tmp["_yp"] = y_hat
for symbol, g in tmp.groupby("symbol"):
    by_symbol[str(symbol)] = metrics(g["_yt"], g["_yp"])
print("per-symbol")
display(pd.DataFrame(by_symbol).T)


per-symbol


,rmse,mae,r2,directional_accuracy
BNBUSDT,0.001287,0.000984,0.254454,0.417361
BTCUSDT,0.001517,0.001094,0.356361,0.425000
ETHUSDT,0.001939,0.001409,0.276405,0.437500
SOLUSDT,0.002050,0.001509,0.333912,0.425990


In [12]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment(EXPERIMENT)
Path("models").mkdir(exist_ok=True)

def log_sk_model(model, registered_name=None):
    kwargs = {
        "sk_model": model,
        "name": "model",
        "serialization_format": "cloudpickle",
    }
    if registered_name:
        kwargs["registered_model_name"] = registered_name
    mlflow.sklearn.log_model(**kwargs)

for name, scores in leaderboard.items():
    with mlflow.start_run(run_name=name):
        mlflow.log_params({
            "model": name,
            "horizon": "24h",
            "holdout_days": HOLDOUT_DAYS,
            "n_trials": N_TRIALS,
        })
        mlflow.log_metrics(scores)
        if name == "lightgbm":
            mlflow.log_params({f"lgbm_{k}": v for k, v in lgbm_study.best_params.items()})
        if name == "xgboost":
            mlflow.log_params({f"xgb_{k}": v for k, v in xgb_study.best_params.items()})
        if name in fitted:
            log_sk_model(fitted[name])

with mlflow.start_run(run_name=f"register-{winner_name}"):
    mlflow.log_params({"winner": winner_name, "horizon": "24h", "interval": "1h"})
    mlflow.log_metrics(leaderboard[winner_name])
    log_sk_model(winner, registered_name=MODEL_NAME)

joblib.dump(winner, "models/model.joblib")
Path("models/feature_columns.json").write_text(json.dumps(FEATURE_COLUMNS, indent=2), encoding="utf-8")
Path("models/train_metrics.json").write_text(
    json.dumps({"winner": winner_name, "leaderboard": leaderboard, "per_symbol": by_symbol}, indent=2),
    encoding="utf-8",
)
print("Saved models/model.joblib  winner=", winner_name)

2026/08/21 05:17:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/08/21 05:17:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/08/21 05:17:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mec

Saved models/model.joblib  winner= voting


## 5. Download artifacts (Colab → your laptop)

Copy these into the project:

- `models/model.joblib`
- `models/feature_columns.json`
- `models/train_metrics.json`
- `mlruns/` (for MLflow UI)

**MLflow UI (local, after download):**

```bash
mlflow ui --backend-store-uri file:./mlruns
```

Open http://127.0.0.1:5000 → Experiments → `crypto-vol-24h` → Models → `crypto_vol_24h`.


In [13]:
from pathlib import Path
from google.colab import files
import shutil

if Path("mlruns").exists():
    shutil.make_archive("mlruns", "zip", "mlruns")
    files.download("mlruns.zip")
else:
    print("No mlruns/ folder (ok if artifacts are only in mlflow.db)")

if Path("mlflow.db").exists():
    files.download("mlflow.db")
else:
    print("mlflow.db not found — run the MLflow cell first")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>